In [ ]:
# Music Therapy Statistical Analysis Examples
# Comprehensive analysis workflow for Traditional Chinese Music therapy pilot study

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📊 Music Therapy Statistical Analysis Examples")
print("=" * 50)

# Import analysis tools
import sys
sys.path.append('../src')
from src.analysis_tools import SmallNAnalyzer
from src.session_runner import SessionDataManager, create_sample_data

# ## 1. Data Preparation and Loading

print("📁 Step 1: Loading Session Data")
print("-" * 30)

# Try to load real session data, fall back to sample data
try:
    session_data = pd.read_csv("../data/session_data/session_log.csv")
    print(f"✅ Loaded real session data: {len(session_data)} responses")
    data_source = "real"
except FileNotFoundError:
    print("📝 Real session data not found, creating sample data for demonstration...")
    session_data = create_sample_data(
        num_participants=4,
        sessions_per_participant=3,
        tracks_per_session=6
    )
    print(f"✅ Created sample data: {len(session_data)} responses")
    data_source = "sample"

# Display basic information
print(f"\n🔍 Dataset Overview:")
print(f"   Participants: {session_data['participant_id'].nunique()}")
print(f"   Sessions: {session_data['session_id'].nunique()}")
print(f"   Total responses: {len(session_data)}")
print(f"   Date range: {session_data['session_date'].min()} to {session_data['session_date'].max()}")

# Show condition distribution
condition_counts = session_data['condition_type'].value_counts()
print(f"\n📊 Condition Distribution:")
for condition, count in condition_counts.items():
    print(f"   {condition.title()}: {count} tracks ({count/len(session_data)*100:.1f}%)")

# Display sample of data
print(f"\n📋 Sample Session Data:")
display_cols = ['participant_id', 'track_title', 'condition_type', 'era', 'mood',
                'engagement_score', 'mood_response_score', 'agitation_score']
sample_display = session_data[display_cols].head(8)
print(sample_display.to_string(index=False))

# ## 2. Initialize Statistical Analyzer

print(f"\n🔧 Step 2: Initializing Statistical Analyzer")
print("-" * 45)

try:
    analyzer = SmallNAnalyzer(session_data)
    print("✅ SmallNAnalyzer initialized successfully")
    print(f"   Data prepared: {len(analyzer.data)} valid observations")
    print(f"   Participants included: {analyzer.data['participant_id'].nunique()}")
except Exception as e:
    print(f"❌ Error initializing analyzer: {e}")
    exit()

# Show data preparation results
print(f"\n📊 Data Preparation Summary:")
print(f"   Original responses: {len(session_data)}")
print(f"   Valid responses: {len(analyzer.data)}")
print(f"   Excluded responses: {len(session_data) - len(analyzer.data)}")

# ## 3. Descriptive Statistics Analysis

print(f"\n📈 Step 3: Descriptive Statistics")
print("-" * 35)

# Basic descriptive statistics
outcome_measures = ['engagement_score', 'mood_response_score', 'agitation_score', 'positive_response']
desc_stats = analyzer.data[outcome_measures].describe()
print(f"📊 Descriptive Statistics:")
print(desc_stats.round(2))

# By condition analysis
print(f"\n🔀 By Condition Analysis:")
by_condition = analyzer.data.groupby('condition_type')[outcome_measures].agg(['mean', 'std', 'count']).round(2)
print(by_condition)

# Individual participant summaries
print(f"\n👥 Individual Participant Summaries:")
individual_stats = analyzer.data.groupby('participant_id')[outcome_measures].agg(['mean', 'count']).round(2)
print(individual_stats)

# ## 4. Effect Size Calculations

print(f"\n📏 Step 4: Effect Size Analysis")
print("-" * 32)

# Calculate effect sizes
effect_sizes = analyzer.calculate_effect_sizes()

if effect_sizes:
    print(f"✅ Effect sizes calculated for {len(effect_sizes)} measures")

    print(f"\n📊 Effect Size Summary:")
    effect_df = pd.DataFrame(effect_sizes).T
    effect_display = effect_df[['cliffs_delta', 'hedges_g', 'interpretation']]
    print(effect_display.round(3))

    # Interpret findings
    print(f"\n🔍 Effect Size Interpretation:")
    for measure, results in effect_sizes.items():
        cliffs_d = results['cliffs_delta']
        interpretation = results['interpretation']
        direction = "favoring AI-generated" if cliffs_d > 0 else "favoring original" if cliffs_d < 0 else "no clear preference"

        print(f"   {measure.replace('_', ' ').title()}:")
        print(f"      Cliff's Δ = {cliffs_d:.3f} ({interpretation} effect, {direction})")

    # Create effect size visualization
    fig_effects = go.Figure()

    measures = list(effect_sizes.keys())
    cliffs_deltas = [effect_sizes[m]['cliffs_delta'] for m in measures]
    interpretations = [effect_sizes[m]['interpretation'] for m in measures]

    colors = ['green' if d > 0.1 else 'red' if d < -0.1 else 'gray' for d in cliffs_deltas]

    fig_effects.add_trace(go.Bar(
        x=measures,
        y=cliffs_deltas,
        text=[f"{d:.3f}<br>({i})" for d, i in zip(cliffs_deltas, interpretations)],
        textposition='auto',
        marker_color=colors,
        name="Cliff's Delta"
    ))

    fig_effects.update_layout(
        title="Effect Sizes: AI-Generated vs Original Music",
        xaxis_title="Outcome Measure",
        yaxis_title="Cliff's Delta",
        yaxis=dict(range=[-1, 1])
    )

    # Add reference lines
    fig_effects.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)
    fig_effects.add_hline(y=0.147, line_dash="dot", line_color="blue", opacity=0.5,
                         annotation_text="Small effect")
    fig_effects.add_hline(y=0.33, line_dash="dot", line_color="orange", opacity=0.5,
                         annotation_text="Medium effect")

    fig_effects.show()

else:
    print("❌ Could not calculate effect sizes (insufficient data)")

# ## 5. Statistical Significance Testing

print(f"\n🧮 Step 5: Statistical Significance Testing")
print("-" * 42)

# Perform statistical tests
statistical_tests = analyzer.perform_statistical_tests()

if statistical_tests:
    print(f"✅ Statistical tests completed for {len(statistical_tests)} measures")

    print(f"\n📊 Statistical Test Results:")
    test_summary = []

    for measure, results in statistical_tests.items():
        test_summary.append({
            'Measure': measure.replace('_', ' ').title(),
            'Test': results['test_type'].replace('_', ' ').title(),
            'p-value': f"{results['p_value']:.4f}",
            'Significant': '✓' if results['significant'] else '✗',
            'Original Mean': f"{results['original_mean']:.2f}",
            'Generated Mean': f"{results['generated_mean']:.2f}"
        })

    test_df = pd.DataFrame(test_summary)
    print(test_df.to_string(index=False))

    # Detailed interpretation
    print(f"\n🔍 Statistical Test Interpretation:")
    significant_results = []
    for measure, results in statistical_tests.items():
        p_val = results['p_value']
        significant = results['significant']
        orig_mean = results['original_mean']
        gen_mean = results['generated_mean']

        print(f"\n   {measure.replace('_', ' ').title()}:")
        print(f"      Test: {results['test_type'].replace('_', ' ').title()}")
        print(f"      p-value: {p_val:.4f}")
        print(f"      Significant: {'Yes' if significant else 'No'} (α = 0.05)")
        print(f"      Original mean: {orig_mean:.2f}")
        print(f"      Generated mean: {gen_mean:.2f}")
        print(f"      Difference: {gen_mean - orig_mean:+.2f}")

        if significant:
            significant_results.append(measure)

    if significant_results:
        print(f"\n🎯 Significant findings detected in: {', '.join([m.replace('_', ' ') for m in significant_results])}")
    else:
        print(f"\n📝 No statistically significant differences found (pilot study expected)")

else:
    print("❌ Could not perform statistical tests (insufficient data)")

# ## 6. Individual Participant Analysis

print(f"\n👤 Step 6: Individual Participant Patterns")
print("-" * 40)

# Analyze individual patterns
individual_patterns = analyzer.analyze_individual_patterns()

if individual_patterns:
    print(f"✅ Individual analysis completed for {len(individual_patterns)} participants")

    # Create individual summary table
    individual_summary = []
    for participant, pattern in individual_patterns.items():
        summary = {
            'Participant': participant,
            'Sessions': pattern['total_sessions'],
            'Tracks': pattern['total_tracks'],
            'Preferred': pattern.get('preferred_condition', 'Unknown').title(),
            'Preference Strength': f"{pattern.get('preference_strength', 0):.2f}",
            'Favorite Era': pattern.get('favorite_era', 'Unknown'),
            'Favorite Mood': pattern.get('favorite_mood', 'Unknown')
        }
        individual_summary.append(summary)

    individual_df = pd.DataFrame(individual_summary)
    print(f"\n📊 Individual Participant Summary:")
    print(individual_df.to_string(index=False))

    # Preference distribution
    preferences = [p.get('preferred_condition', 'Unknown') for p in individual_patterns.values()]
    pref_counts = pd.Series(preferences).value_counts()

    print(f"\n🎯 Preference Distribution:")
    for pref, count in pref_counts.items():
        print(f"   {pref.title()}: {count} participants ({count/len(individual_patterns)*100:.1f}%)")

    # Individual trajectory visualization
    fig_individual = go.Figure()

    participants = list(individual_patterns.keys())
    for i, participant in enumerate(participants):
        participant_data = analyzer.data[analyzer.data['participant_id'] == participant]
        participant_data = participant_data.sort_values(['session_date', 'track_order'])

        # Create track index for x-axis
        track_indices = range(len(participant_data))

        colors = ['blue' if cond == 'original' else 'red'
                 for cond in participant_data['condition_type']]

        fig_individual.add_trace(go.Scatter(
            x=track_indices,
            y=participant_data['engagement_score'],
            mode='lines+markers',
            name=f'{participant}',
            line=dict(color=px.colors.qualitative.Set1[i % len(px.colors.qualitative.Set1)]),
            marker=dict(
                color=colors,
                size=8,
                line=dict(width=1, color='white')
            )
        ))

    fig_individual.update_layout(
        title="Individual Participant Engagement Trajectories",
        xaxis_title="Track Number",
        yaxis_title="Engagement Score",
        legend_title="Participant"
    )
    fig_individual.show()

else:
    print("❌ Could not analyze individual patterns")

# ## 7. Musical Characteristics Analysis

print(f"\n🎵 Step 7: Musical Characteristics Analysis")
print("-" * 42)

# Era preferences
era_engagement = analyzer.data.groupby('era')['engagement_score'].agg(['mean', 'std', 'count']).round(2)
print(f"📊 Engagement by Era:")
print(era_engagement)

# Mood preferences
mood_engagement = analyzer.data.groupby('mood')['engagement_score'].agg(['mean', 'std', 'count']).round(2)
print(f"\n😊 Engagement by Mood:")
print(mood_engagement)

# Instrument analysis (if available)
if 'instruments' in analyzer.data.columns:
    # Extract most common instruments
    all_instruments = []
    for instruments in analyzer.data['instruments']:
        if isinstance(instruments, str) and instruments:
            all_instruments.extend([inst.strip() for inst in instruments.split(',')])

    if all_instruments:
        instrument_counts = pd.Series(all_instruments).value_counts()
        print(f"\n🎻 Most Common Instruments:")
        print(instrument_counts.head())

# Create musical characteristics visualization
fig_musical = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Engagement by Era', 'Engagement by Mood',
                   'Mood Response by Era', 'Agitation by Mood'),
    vertical_spacing=0.1
)

# Era engagement
fig_musical.add_trace(
    go.Bar(x=era_engagement.index, y=era_engagement['mean'],
          error_y=dict(type='data', array=era_engagement['std']),
          name='Era Engagement', showlegend=False),
    row=1, col=1
)

# Mood engagement
fig_musical.add_trace(
    go.Bar(x=mood_engagement.index, y=mood_engagement['mean'],
          error_y=dict(type='data', array=mood_engagement['std']),
          name='Mood Engagement', showlegend=False),
    row=1, col=2
)

# Mood response by era
mood_by_era = analyzer.data.groupby('era')['mood_response_score'].mean()
fig_musical.add_trace(
    go.Bar(x=mood_by_era.index, y=mood_by_era.values,
          name='Mood by Era', showlegend=False),
    row=2, col=1
)

# Agitation by mood
agitation_by_mood = analyzer.data.groupby('mood')['agitation_score'].mean()
fig_musical.add_trace(
    go.Bar(x=agitation_by_mood.index, y=agitation_by_mood.values,
          name='Agitation by Mood', showlegend=False),
    row=2, col=2
)

fig_musical.update_layout(
    title="Musical Characteristics Analysis",
    height=600
)
fig_musical.show()

# ## 8. Time Series Analysis (if applicable)

if 'session_date' in analyzer.data.columns:
    print(f"\n📅 Step 8: Time Series Analysis")
    print("-" * 30)

    # Convert to datetime
    analyzer.data['session_date'] = pd.to_datetime(analyzer.data['session_date'])

    # Daily averages
    daily_stats = analyzer.data.groupby(['session_date', 'condition_type']).agg({
        'engagement_score': 'mean',
        'mood_response_score': 'mean',
        'agitation_score': 'mean'
    }).reset_index()

    print(f"📊 Daily Statistics:")
    print(daily_stats.head())

    # Time series visualization
    fig_timeseries = make_subplots(
        rows=3, cols=1,
        subplot_titles=('Engagement Over Time', 'Mood Response Over Time', 'Agitation Over Time'),
        vertical_spacing=0.1
    )

    measures = ['engagement_score', 'mood_response_score', 'agitation_score']
    colors = {'original': 'blue', 'generated': 'red'}

    for i, measure in enumerate(measures, 1):
        for condition in ['original', 'generated']:
            condition_data = daily_stats[daily_stats['condition_type'] == condition]

            if len(condition_data) > 0:
                fig_timeseries.add_trace(
                    go.Scatter(
                        x=condition_data['session_date'],
                        y=condition_data[measure],
                        mode='lines+markers',
                        name=f'{condition.title()}',
                        line=dict(color=colors[condition]),
                        showlegend=(i==1)
                    ),
                    row=i, col=1
                )

    fig_timeseries.update_layout(
        title="Response Trends Over Time",
        height=800
    )
    fig_timeseries.show()

# ## 9. Comprehensive Visualization Dashboard

print(f"\n📊 Step 9: Creating Comprehensive Dashboard")
print("-" * 44)

# Create main comparison visualization
fig_dashboard = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Engagement Comparison', 'Mood Response Comparison', 'Agitation Comparison',
                   'Effect Sizes', 'Individual Preferences', 'Era Preferences'),
    specs=[[{}, {}, {}],
           [{}, {"type": "domain"}, {}]]
)

# Outcome comparisons (box plots)
measures_info = [
    ('engagement_score', 'Engagement'),
    ('mood_response_score', 'Mood Response'),
    ('agitation_score', 'Agitation')
]

for i, (measure, title) in enumerate(measures_info):
    for condition in ['original', 'generated']:
        data = analyzer.data[analyzer.data['condition_type'] == condition][measure]
        color = 'lightblue' if condition == 'original' else 'lightcoral'

        fig_dashboard.add_trace(
            go.Box(y=data, name=condition.title(),
                  boxpoints='all', jitter=0.3,
                  marker_color=color,
                  showlegend=(i==0)),
            row=1, col=i+1
        )

# Effect sizes (if available)
if effect_sizes:
    measures = list(effect_sizes.keys())[:3]  # Top 3 measures
    cliffs_deltas = [effect_sizes[m]['cliffs_delta'] for m in measures]

    fig_dashboard.add_trace(
        go.Bar(x=measures, y=cliffs_deltas,
              marker_color=['green' if d > 0 else 'red' for d in cliffs_deltas],
              showlegend=False),
        row=2, col=1
    )

# Individual preferences (pie chart)
if individual_patterns:
    preferences = [p.get('preferred_condition', 'Unknown') for p in individual_patterns.values()]
    pref_counts = pd.Series(preferences).value_counts()

    fig_dashboard.add_trace(
        go.Pie(labels=pref_counts.index, values=pref_counts.values,
              name="Preferences", showlegend=False),
        row=2, col=2
    )

# Era preferences
era_scores = analyzer.data.groupby('era')['engagement_score'].mean().sort_values(ascending=False)
fig_dashboard.add_trace(
    go.Bar(x=era_scores.index, y=era_scores.values,
          marker_color='skyblue', showlegend=False),
    row=2, col=3
)

fig_dashboard.update_layout(
    title="Music Therapy Analysis Dashboard",
    height=800,
    showlegend=True
)
fig_dashboard.show()

# ## 10. Generate Comprehensive Report

print(f"\n📄 Step 10: Generating Analysis Report")
print("-" * 38)

try:
    # Generate full HTML report
    report_path = analyzer.generate_report("music_therapy_analysis_report.html")
    print(f"✅ Comprehensive report generated: {report_path}")

    # Create plots directory
    plots_dir = "../results/plots"
    import os
    os.makedirs(plots_dir, exist_ok=True)

    # Generate and save visualizations
    plots = analyzer.create_visualizations(plots_dir)
    print(f"✅ Visualizations saved to: {plots_dir}")

    for plot_name, plot_path in plots.items():
        print(f"   📊 {plot_name}: {plot_path}")

except Exception as e:
    print(f"❌ Error generating report: {e}")

# ## 11. Export Results for Paper

print(f"\n📝 Step 11: Preparing Results for Publication")
print("-" * 43)

# Create summary for paper
paper_results = {
    'study_info': {
        'total_participants': analyzer.data['participant_id'].nunique(),
        'total_sessions': analyzer.data['session_id'].nunique(),
        'total_responses': len(analyzer.data),
        'data_source': data_source,
        'analysis_date': datetime.now().isoformat()
    },
    'descriptive_statistics': {
        'by_condition': by_condition.to_dict() if 'by_condition' in locals() else {},
        'overall': desc_stats.to_dict() if 'desc_stats' in locals() else {}
    },
    'effect_sizes': effect_sizes if effect_sizes else {},
    'statistical_tests': statistical_tests if statistical_tests else {},
    'individual_patterns': {
        'total_analyzed': len(individual_patterns) if individual_patterns else 0,
        'preference_distribution': pref_counts.to_dict() if 'pref_counts' in locals() else {},
        'individual_summaries': individual_patterns if individual_patterns else {}
    },
    'musical_characteristics': {
        'era_engagement': era_engagement.to_dict() if 'era_engagement' in locals() else {},
        'mood_engagement': mood_engagement.to_dict() if 'mood_engagement' in locals() else {}
    }
}

# Save results for paper
results_path = "../results/paper_results.json"
os.makedirs("../results", exist_ok=True)
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(paper_results, f, indent=2, ensure_ascii=False, default=str)

print(f"✅ Paper results saved to: {results_path}")

# Create tables for paper
if statistical_tests:
    # Statistical results table
    stats_table = pd.DataFrame([
        {
            'Measure': measure.replace('_', ' ').title(),
            'Original_Mean': f"{results['original_mean']:.2f}",
            'Generated_Mean': f"{results['generated_mean']:.2f}",
            'Test_Statistic': f"{results['statistic']:.2f}",
            'P_Value': f"{results['p_value']:.4f}",
            'Significant': 'Yes' if results['significant'] else 'No'
        }
        for measure, results in statistical_tests.items()
    ])

    stats_table_path = "../results/statistical_results_table.csv"
    stats_table.to_csv(stats_table_path, index=False)
    print(f"📊 Statistical results table: {stats_table_path}")

if effect_sizes:
    # Effect sizes table
    effect_table = pd.DataFrame([
        {
            'Measure': measure.replace('_', ' ').title(),
            'Cliffs_Delta': f"{results['cliffs_delta']:.3f}",
            'Hedges_g': f"{results['hedges_g']:.3f}",
            'Interpretation': results['interpretation'].title()
        }
        for measure, results in effect_sizes.items()
    ])

    effect_table_path = "../results/effect_sizes_table.csv"
    effect_table.to_csv(effect_table_path, index=False)
    print(f"📏 Effect sizes table: {effect_table_path}")

# ## 12. Analysis Summary and Interpretation

print(f"\n🎯 Analysis Summary and Key Findings")
print("=" * 40)

print(f"📊 Study Overview:")
print(f"   • Participants analyzed: {analyzer.data['participant_id'].nunique()}")
print(f"   • Total track responses: {len(analyzer.data)}")
print(f"   • Conditions compared: Original vs AI-Generated Traditional Chinese Music")
print(f"   • Primary outcomes: Engagement, Mood Response, Agitation")

if effect_sizes:
    print(f"\n📏 Effect Size Findings:")
    for measure, results in effect_sizes.items():
        direction = "AI-generated" if results['cliffs_delta'] > 0 else "Original"
        magnitude = results['interpretation']
        print(f"   • {measure.replace('_', ' ').title()}: {magnitude} effect favoring {direction}")

if statistical_tests:
    print(f"\n🧮 Statistical Significance:")
    significant_count = sum(1 for r in statistical_tests.values() if r['significant'])
    print(f"   • Significant differences found: {significant_count}/{len(statistical_tests)} measures")

    if significant_count > 0:
        sig_measures = [m for m, r in statistical_tests.items() if r['significant']]
        print(f"   • Significant measures: {', '.join([m.replace('_', ' ') for m in sig_measures])}")

if individual_patterns:
    print(f"\n👥 Individual Variation:")
    if 'pref_counts' in locals():
        for condition, count in pref_counts.items():
            print(f"   • Participants preferring {condition}: {count}")
    print(f"   • Individual differences highlight need for personalized approaches")

print(f"\n🎵 Musical Characteristics:")
if 'era_engagement' in locals():
    best_era = era_engagement['mean'].idxmax()
    print(f"   • Highest engagement era: {best_era}")
if 'mood_engagement' in locals():
    best_mood = mood_engagement['mean'].idxmax()
    print(f"   • Highest engagement mood: {best_mood}")

print(f"\n🔬 Research Implications:")
print(f"   • This pilot study provides preliminary evidence for AI-augmented music therapy")
print(f"   • Individual preferences vary, supporting personalized intervention approaches")
print(f"   • Cultural authenticity remains important in AI-generated music")
print(f"   • Larger studies needed to confirm these preliminary findings")

print(f"\n📝 Next Steps for Research:")
print(f"   • Increase sample size for stronger statistical power")
print(f"   • Conduct longer-term follow-up studies")
print(f"   • Investigate individual factors predicting music preferences")
print(f"   • Validate cultural authenticity with expert assessment")
print(f"   • Explore physiological response measures")

print(f"\n✅ Analysis Complete!")
print(f"📁 Results exported to: ../results/")
print(f"📊 Ready for paper writing and submission")

# ## Bonus: Quick Analysis Function for Ongoing Studies

def quick_analysis_update(data_path="../data/session_data/session_log.csv"):
    """
    Quick analysis function for ongoing studies.
    Call this function to get immediate insights from new session data.
    """
    try:
        data = pd.read_csv(data_path)
        analyzer = SmallNAnalyzer(data)

        print(f"🔄 Quick Analysis Update")
        print(f"   Participants: {data['participant_id'].nunique()}")
        print(f"   Responses: {len(data)}")

        # Quick effect sizes
        effects = analyzer.calculate_effect_sizes()
        if effects:
            print(f"   Key findings:")
            for measure, result in effects.items():
                if abs(result['cliffs_delta']) > 0.1:  # Notable effects
                    direction = "AI" if result['cliffs_delta'] > 0 else "Original"
                    print(f"     {measure}: {result['interpretation']} effect → {direction}")

        return analyzer

    except Exception as e:
        print(f"❌ Quick analysis failed: {e}")
        return None

print(f"\n💡 Use quick_analysis_update() for ongoing analysis as new data arrives!")